In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split as train_test_split_surprise
from surprise.model_selection import cross_validate
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as train_test_split_sklearn

C:\Users\JWinn01\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
point = (
    '../data/raw/movies.csv'
)
datos_movies = pd.read_csv(point, sep=',')
datos_movies.head(3)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [4]:
point = (
    '../data/raw/ratings.csv'
)
datos_ratings = pd.read_csv(point, sep=',')
datos_ratings.head(3)

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828


In [5]:
datos_movies["genres"] = datos_movies["genres"].str.split("|")

df_exploded = datos_movies.explode('genres')
genre_dummies = pd.get_dummies(df_exploded['genres'])

df_combined = pd.concat([df_exploded[['movieId', 'title']], genre_dummies], axis=1)

In [6]:
df_final_movies = df_combined.groupby(['movieId', 'title'], as_index=False).sum()

In [7]:
df_final_movies

,movieId,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62418,209157,We (2018),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
62419,209159,Window of the Soul (2001),0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
62420,209163,Bad Poems (2018),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
62421,209169,A Girl Thing (2001),1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
df = merged_data = pd.merge(
    datos_ratings,
    df_final_movies,
    on="movieId",
    how="right"
)
df.head()

,userId,movieId,rating,timestamp,title,(no genres listed),Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,2.0,1,3.5,1.141416e+09,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,3.0,1,4.0,1.439472e+09,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,4.0,1,3.0,1.573944e+09,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,5.0,1,4.0,8.586259e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,8.0,1,4.0,8.904925e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
datos_ratings['userId'].nunique()

162541

In [10]:
df_colab  = datos_ratings.groupby('movieId')['rating'].mean().reset_index()

# opcional: renombrar la columna para mayor claridad
mean_ratings = df_colab.rename(columns={'rating': 'rating_medio'})

mean_ratings.head(20)

,movieId,rating_medio
0,1,3.893708
1,2,3.251527
2,3,3.142028
3,4,2.853547
4,5,3.058434
5,6,3.854909
6,7,3.363666
7,8,3.114583
8,9,2.992051
9,10,3.421458


In [11]:
X = mean_ratings
y = mean_ratings

In [12]:
#df_finito = df[['userId', 'movieId', 'rating']]
reader = Reader(rating_scale=(1, 5))
df_finito = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

In [13]:
trainset, testset = train_test_split_surprise(df_finito, test_size=0.2, random_state=42)

In [14]:
df_finito

In [ ]:
# **Modelo KNN basado en ítems**
sim_options = {'name': 'cosine', 'user_based': True}  # Cambia user_based=True para KNN de usuarios
model = SVD(n_factors=50, n_epochs=20)  # Esta linea dedicada para ajustar el consumo de RAM
#model = KNNBasic(sim_options=sim_options) #Errores por tamaño de dataset
model.fit(trainset)

predictions = model.test(testset)#predecimos

In [ ]:
rmse = accuracy.rmse(predictions)
print(f'RMSE: {rmse}')

RMSE: nan
RMSE: nan


In [ ]:

user = str(96737)  # IDs deben ser strings en Surprise.... por alguna razón 
item = str(242)
pred = model.predict(user, item)
print(f'Predicción de usuario {user} para peli {item}: {pred.est}')

Predicción de usuario 196 para ítem 242: 5


In [ ]:
predictions

[Prediction(uid=128631.0, iid=96737, r_ui=4.5, est=5, details={'was_impossible': False}),
 Prediction(uid=41761.0, iid=193, r_ui=1.5, est=5, details={'was_impossible': False}),
 Prediction(uid=121425.0, iid=2348, r_ui=2.0, est=5, details={'was_impossible': False}),
 Prediction(uid=17674.0, iid=1222, r_ui=3.0, est=5, details={'was_impossible': False}),
 Prediction(uid=89415.0, iid=1396, r_ui=1.5, est=5, details={'was_impossible': False}),
 Prediction(uid=106481.0, iid=1204, r_ui=5.0, est=5, details={'was_impossible': False}),
 Prediction(uid=12220.0, iid=180, r_ui=5.0, est=5, details={'was_impossible': False}),
 Prediction(uid=nan, iid=185379, r_ui=nan, est=5, details={'was_impossible': False}),
 Prediction(uid=140711.0, iid=64957, r_ui=4.5, est=5, details={'was_impossible': False}),
 Prediction(uid=33313.0, iid=1952, r_ui=4.0, est=5, details={'was_impossible': False}),
 Prediction(uid=12979.0, iid=55363, r_ui=4.5, est=5, details={'was_impossible': False}),
 Prediction(uid=86170.0, iid=